## Prepare public data for benchmarking

Convert the downloaded AG2P-DISC RData files into CSVs for the external-validation notebook. 

Download and extract each crop's `curated_geno_pheno.rdata` into `data/benchmarks/learnable/raw/<crop>/` first before running this notebook  
Dataset links and download instructions are in the project README.

In [1]:
from pathlib import Path

requirements = next(
    folder / "requirements.txt"
    for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "requirements.txt").is_file()
)

%pip install -r "{requirements}"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import subprocess
import sys

# Locate the project when Jupyter starts in the project root or notebooks folder.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / 'src' / 'external_validation' / 'prepare_ag2p_benchmark.py').is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError('Start Jupyter from the project root or notebooks folder.')

SCRIPT = ROOT / 'src' / 'external_validation' / 'prepare_ag2p_benchmark.py'
RAW_DIR = ROOT / 'data' / 'benchmarks' / 'learnable' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'benchmarks' / 'learnable' / 'processed'

# Remove crop names here to prepare only a subset.
DATASETS = ['commonbean', 'durum_wheat', 'maize', 'rice210', 'soybean']
# Change to True only if existing CSVs may be replaced.
OVERWRITE = False
print(f'Project root: {ROOT}')
print(f'Selected datasets: {", ".join(DATASETS)}')

Project root: c:\Users\frank\OneDrive - Mississippi State University\MADS\Capstone Project\capstone\fwk_capstone_v1.0
Selected datasets: commonbean, durum_wheat, maize, rice210, soybean


### Check inputs and existing outputs

All selected RData files must exist before preparation starts. 

In [3]:
sources = {crop: RAW_DIR / crop / 'curated_geno_pheno.rdata' for crop in DATASETS}
missing = [path for path in sources.values() if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing source files:\n' + '\n'.join(map(str, missing)))

existing = [
    PROCESSED_DIR / f'{kind}_external_{crop}.csv'
    for crop in DATASETS for kind in ('genotype', 'phenotype')
    if (PROCESSED_DIR / f'{kind}_external_{crop}.csv').exists()
]
if existing and not OVERWRITE:
    raise FileExistsError('Existing outputs will not be overwritten:\n' + '\n'.join(map(str, existing)))

for crop, source in sources.items():
    print(f'{crop}: {source}')
print(f'Output directory: {PROCESSED_DIR}')

commonbean: c:\Users\frank\OneDrive - Mississippi State University\MADS\Capstone Project\capstone\fwk_capstone_v1.0\data\benchmarks\learnable\raw\commonbean\curated_geno_pheno.rdata
durum_wheat: c:\Users\frank\OneDrive - Mississippi State University\MADS\Capstone Project\capstone\fwk_capstone_v1.0\data\benchmarks\learnable\raw\durum_wheat\curated_geno_pheno.rdata
maize: c:\Users\frank\OneDrive - Mississippi State University\MADS\Capstone Project\capstone\fwk_capstone_v1.0\data\benchmarks\learnable\raw\maize\curated_geno_pheno.rdata
rice210: c:\Users\frank\OneDrive - Mississippi State University\MADS\Capstone Project\capstone\fwk_capstone_v1.0\data\benchmarks\learnable\raw\rice210\curated_geno_pheno.rdata
soybean: c:\Users\frank\OneDrive - Mississippi State University\MADS\Capstone Project\capstone\fwk_capstone_v1.0\data\benchmarks\learnable\raw\soybean\curated_geno_pheno.rdata
Output directory: c:\Users\frank\OneDrive - Mississippi State University\MADS\Capstone Project\capstone\fwk_ca

### Prepare input CSVs

The script runs once per selected crop

In [4]:
# Recheck immediately before writing, including if this cell is rerun.
existing = [
    PROCESSED_DIR / f'{kind}_external_{crop}.csv'
    for crop in DATASETS for kind in ('genotype', 'phenotype')
    if (PROCESSED_DIR / f'{kind}_external_{crop}.csv').exists()
]
if existing and not OVERWRITE:
    raise FileExistsError('Existing outputs will not be overwritten:\n' + '\n'.join(map(str, existing)))

for crop, source in sources.items():
    print(f'Preparing {crop}...', flush=True)
    subprocess.run([sys.executable, str(SCRIPT), str(source), crop], check=True)

print('Preparation complete.')
print('Run `02_external_data_validation.ipynb` next.')

Preparing commonbean...
Preparing durum_wheat...
Preparing maize...
Preparing rice210...
Preparing soybean...
Preparation complete.
Run `02_external_data_validation.ipynb` next.
